In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q accelerate
!pip install -q scikit-learn
!pip install -q transformers qwen-vl-utils
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [2]:
import os
import gc
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from datasets import load_dataset
import transformers
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
import bitsandbytes as bnb 
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm 
import warnings
import ctypes

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 0. THIẾT LẬP HỆ THỐNG & MULTI-GPU CẤP THẤP
# ==========================================
transformers.PreTrainedModel.all_tied_weights_keys = {}
transformers.logging.set_verbosity_error()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
torch.backends.cudnn.benchmark = True 

# Ép toàn bộ lưu trữ sang RAM máy tính (CPU) để các GPU chuyên tâm cho huấn luyện
DEVICE_STORAGE = torch.device("cpu")
print(f"🚀 Phân bổ hệ thống chống OOM: \n - Huấn luyện: Phân tán tự động qua các GPU (device_map='auto') \n - Lưu trữ & Tổng hợp (Aggregator): Trên RAM {DEVICE_STORAGE}")

# CẤU HÌNH SỐ LƯỢNG CLIENT LÀ 5
NUM_CLIENTS = 5
runName = f"FedMVQA_VQARAD_InternVL3.5_FFT_448_Alpha1_R1_E2_{NUM_CLIENTS}Clients_DualGPU"
kaggleBaseDir = "/kaggle/working/Fed-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "vqarad")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)

for d in [dataDir, logDir, checkpointDir]: os.makedirs(d, exist_ok=True)
logPath = os.path.join(logDir, f"{runName}_log.csv")
checkpointPath = os.path.join(checkpointDir, "best_fed_mvqa_model.pt")

# ==========================================
# 1. CÔNG CỤ QUẢN LÝ BỘ NHỚ
# ==========================================
def force_garbage_collect():
    gc.collect()
    if torch.cuda.is_available():
        # Đồng bộ và xóa cache trên tất cả GPU đang có
        for i in range(torch.cuda.device_count()):
            torch.cuda.synchronize(i)
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except: pass

def getPeakReservedMemory():
    memory_info = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            memory_info.append(f"GPU{i}: {torch.cuda.memory_reserved(i)/(1024**3):.1f}GB")
    return " | ".join(memory_info)

def processImage(pilImg, targetSize=448):
    if pilImg.mode != "RGB": pilImg = pilImg.convert("RGB")
    imageTransform = T.Compose([
        T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])
    return imageTransform(pilImg).unsqueeze(0)

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC (TỰ ĐỘNG CHIA LÊN CÁC GPU)
# ==========================================
print(f"--- KHỞI TẠO FED-FFT (INTERNVL3.5-2B, 448x448, FP16 - {NUM_CLIENTS} CLIENTS) ---")
force_garbage_collect()

modelId = "OpenGVLab/InternVL3_5-2B-Instruct"

# Sử dụng device_map="auto" để cắt đôi Model chia cho GPU 0 và GPU 1
baseModel = AutoModel.from_pretrained(
    modelId, 
    torch_dtype=torch.float16, 
    trust_remote_code=True, 
    low_cpu_mem_usage=True,
    device_map="auto" 
)
tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)

# ---------------------------------------------------------
# 🔥 BẢN VÁ LỖI MULTI-GPU (DEVICE MAP) CHO INTERNVL
# Tránh lỗi: "indices should be either on cpu or on the same device"
if hasattr(baseModel, 'extract_feature'):
    original_extract = baseModel.extract_feature
    def safe_extract(*args, **kwargs):
        # Lấy feature từ vision_model (có thể đang ở GPU 1)
        features = original_extract(*args, **kwargs)
        # Ép feature trả về cùng GPU 0 với biến image_flags để PyTorch tính toán được
        return features.to(baseModel.device)
    baseModel.extract_feature = safe_extract
# ---------------------------------------------------------

baseModel.config.use_cache = False
if hasattr(baseModel, 'enable_input_require_grads'): baseModel.enable_input_require_grads()
try: baseModel.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except: pass
if hasattr(baseModel, 'language_model'): baseModel.language_model.gradient_checkpointing_enable()
if hasattr(baseModel, 'vision_model'): baseModel.vision_model.gradient_checkpointing_enable()

for p in baseModel.parameters(): p.requires_grad = True

totalModelParams = sum(p.numel() for p in baseModel.parameters())
# Lưu globalWeights HOÀN TOÀN TRÊN RAM CPU để tránh chiếm dụng VRAM của GPU
globalWeights = {k: v.detach().to(DEVICE_STORAGE).clone() for k, v in baseModel.state_dict().items()}
force_garbage_collect()

# ==========================================
# 3. TRÍCH XUẤT ĐỘNG ID TOKEN HÌNH ẢNH 
# ==========================================
print("Đang trích xuất tự động Image Context ID...")
interceptedIds = None
originalForward = baseModel.forward

def hookForward(*args, **kwargs):
    global interceptedIds
    if 'input_ids' in kwargs and kwargs['input_ids'] is not None: interceptedIds = kwargs['input_ids'].detach().cpu()
    elif len(args) > 1 and args[1] is not None: interceptedIds = args[1].detach().cpu()
    raise RuntimeError("STOP_INTERCEPT")

baseModel.forward = hookForward
try:
    # Gửi tensor dummy vào thiết bị gốc của mô hình
    baseModel.chat(tokenizer, torch.zeros((1, 3, 448, 448), dtype=torch.float16).to(baseModel.device), "", generation_config=dict(max_new_tokens=1))
except: pass
finally: baseModel.forward = originalForward

imgContextId = int(torch.bincount(interceptedIds.view(-1)).argmax().item()) if interceptedIds is not None else getattr(baseModel, 'img_context_token_id', tokenizer.convert_tokens_to_ids(''))
print(f"✅ Image Context ID: {imgContextId}")

# ==========================================
# 4. CHUẨN BỊ DỮ LIỆU VQA-RAD
# ==========================================
dataset = load_dataset("flaviagiammarino/vqa-rad", cache_dir=dataDir)
shuffledTrain = dataset["train"].shuffle(seed=42)
testDataset = dataset["test"]

def partitionDataNonIidDirichlet(datasetInput, numClients, alpha=1.0):
    labels = [str(item["answer"]).strip().lower() for item in datasetInput]
    uniqueLabels = list(set(labels))
    clientIndices = {i: [] for i in range(numClients)}
    for label in uniqueLabels:
        labelIdxs = [i for i, l in enumerate(labels) if l == label]
        np.random.shuffle(labelIdxs)
        splits = np.split(labelIdxs, (np.cumsum(np.random.dirichlet([alpha] * numClients))[:-1] * len(labelIdxs)).astype(int))
        for i in range(numClients): clientIndices[i].extend(splits[i].tolist())
    return [datasetInput.select(np.random.permutation(clientIndices[i])) for i in range(numClients)]

clientDatasets = partitionDataNonIidDirichlet(shuffledTrain, NUM_CLIENTS, alpha=1.0)
clientNames = [chr(65+i) for i in range(NUM_CLIENTS)] # A, B, C, D, E
dataSizes = [len(ds) for ds in clientDatasets]

print(f"Phân bổ dữ liệu cho {NUM_CLIENTS} Clients:")
for i, name in enumerate(clientNames):
    print(f" - Client {name}: {dataSizes[i]} mẫu")

systemInstruction = "You are an expert medical diagnostic assistant. Your task is to answer clinical questions based EXCLUSIVELY on the provided medical image.\nStrictly follow these rules:\n1. Analyze the provided image.\n2. Provide a concise and definitive answer (1-3 words if possible)."

def customCollateFn(batch):
    pixelValuesList, inputIdsList, labelsList = [], [], []
    for item in batch:
        img = item["image"].convert("RGB") 
        q, a = item["question"], str(item["answer"]).strip()
        
        promptPart1 = f"{systemInstruction}\nUser: "
        promptPart2 = f"\nQuestion: {q}\nFinal Answer:\nAssistant: "

        part1Ids = tokenizer(promptPart1, return_tensors="pt", add_special_tokens=True).input_ids[0]
        part2Ids = tokenizer(promptPart2, return_tensors="pt", add_special_tokens=False).input_ids[0]
        answerIds = tokenizer(a + getattr(tokenizer, 'eos_token', ''), return_tensors="pt", add_special_tokens=False).input_ids[0]
        
        imgTokens = torch.full((256,), imgContextId, dtype=torch.long)
        inputIds = torch.cat([part1Ids, imgTokens, part2Ids, answerIds], dim=0)
        
        lbls = inputIds.clone()
        lbls[:len(part1Ids) + 256 + len(part2Ids)] = -100 
        
        inputIdsList.append(inputIds); labelsList.append(lbls); pixelValuesList.append(processImage(img))

    maxLen = max(len(ids) for ids in inputIdsList)
    padId = getattr(tokenizer, 'pad_token_id', getattr(tokenizer, 'eos_token_id', 0)) or 0
    paddedInputIds, paddedLabels, attentionMask = [], [], []
    
    for ids, lbls in zip(inputIdsList, labelsList):
        padLen = maxLen - len(ids)
        paddedInputIds.append(torch.cat([ids, torch.full((padLen,), padId, dtype=torch.long)]))
        paddedLabels.append(torch.cat([lbls, torch.full((padLen,), -100, dtype=torch.long)]))
        attentionMask.append(torch.cat([torch.ones(len(ids), dtype=torch.long), torch.zeros(padLen, dtype=torch.long)]))

    return {
        "input_ids": torch.stack(paddedInputIds), "labels": torch.stack(paddedLabels),
        "attention_mask": torch.stack(attentionMask), "pixel_values": torch.cat(pixelValuesList, dim=0).to(torch.float16),
        "image_flags": torch.ones((len(batch), 1), dtype=torch.long) 
    }

trainLoaders = [DataLoader(ds, batch_size=1, shuffle=True, collate_fn=customCollateFn, num_workers=0) for ds in clientDatasets]

# ==========================================
# 5. HÀM HUẤN LUYỆN ĐA GPU & TỔNG HỢP RAM CPU
# ==========================================
def trainLocalClient(model, dataLoader, clientId, roundIdx, epochs=2, learningRate=1e-5):
    model.train()
    activeLr = learningRate * (0.9 ** (roundIdx - 1))
    
    # Optimizer PagedAdamW8bit sẽ tự động Offload gradient rác vào CPU giúp chống tràn
    optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=activeLr, weight_decay=0.01)
    accumulationSteps = 4 
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(dataLoader) // accumulationSteps)

    print(f" [Client {clientId}] Bắt đầu huấn luyện FFT (PagedAdamW8bit, LR={activeLr:.2e})...")
    startTime = time.time()
    totalLoss = 0

    for epoch in range(epochs):
        epochLoss = 0
        optimizer.zero_grad(set_to_none=True)
        progressLoop = tqdm(dataLoader, desc=f"Client {clientId} | Epoch {epoch+1}/{epochs}", leave=False, dynamic_ncols=True)
        
        for step, batch in enumerate(progressLoop):
            # Cấp dữ liệu cho layer đầu tiên, HuggingFace Accelerate tự truyền qua các GPU tiếp theo
            batch = {k: v.to(model.device, non_blocking=True) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / accumulationSteps
            
            loss.backward()
            
            step_loss = loss.item() * accumulationSteps
            epochLoss += step_loss
            totalLoss += step_loss
            
            del outputs, loss, batch
            
            if (step + 1) % accumulationSteps == 0 or (step + 1) == len(dataLoader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                
            progressLoop.set_postfix(Loss=f"{step_loss:.4f}", LR=f"{scheduler.get_last_lr()[0]:.2e}")
            if step % 30 == 0: force_garbage_collect()
            
        print(f"     Epoch {epoch + 1} hoàn tất | Avg Loss: {epochLoss / len(dataLoader):.4f}")
            
    trainDuration = time.time() - startTime
    avgLoss = totalLoss / (len(dataLoader) * epochs)
    
    del optimizer, scheduler
    model.zero_grad(set_to_none=True) 
    force_garbage_collect()
    return trainDuration, avgLoss, getPeakReservedMemory()

def aggregate_client_weights(model, global_weights, weight_ratio):
    """Cộng dồn trọng số an toàn trên RAM CPU, không chạm vào GPU"""
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                # Đẩy từ các GPU về CPU để làm toán cộng
                temp_weight = v.detach().to(DEVICE_STORAGE, dtype=torch.float32)
                temp_weight.mul_(weight_ratio)
                global_weights[k].add_(temp_weight)
                del temp_weight
    force_garbage_collect()

# ==========================================
# 6. HÀM ĐÁNH GIÁ
# ==========================================
def evaluateGlobalModel(model, datasetInput):
    model.eval()
    print(f"\n [Server] Đang đánh giá trên tập Test ({len(datasetInput)} mẫu)...")
    openPreds, openRefs, closePreds, closeRefs, inferenceTimes = [], [], [], [], []
    
    for item in tqdm(datasetInput, desc="Evaluating", leave=False, dynamic_ncols=True):
        q, gtAnswer = item["question"], str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN")).strip().upper()
        
        promptText = f"{systemInstruction}\n[Target Case Image]\n\n\nCurrent Question: {q}\nBased on the visual evidence, what is the correct answer?\nFinal Answer: "
        pixelValues = processImage(item["image"]).to(torch.float16).to(model.device)
        
        startTime = time.time()
        with torch.inference_mode():
            aiAnswerRaw = model.chat(tokenizer=tokenizer, pixel_values=pixelValues, question=promptText, generation_config=dict(max_new_tokens=15, do_sample=False))
        inferenceTimes.append(time.time() - startTime)
        aiAnswer = aiAnswerRaw.lower().strip()
        
        if ansType == "CLOSED":
            predClean = 'yes' if 'yes' in aiAnswer and 'no' not in aiAnswer else ('no' if 'no' in aiAnswer and 'yes' not in aiAnswer else ('yes' if aiAnswer.find('yes') < aiAnswer.find('no') else 'no'))
            closePreds.append(predClean)
            closeRefs.append('yes' if 'yes' in gtAnswer else ('no' if 'no' in gtAnswer else gtAnswer))
        else:
            cleanStr = lambda t: " ".join([w for w in re.sub(r'[^\w\s]', '', t.lower()).split() if w not in {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}])
            openPreds.append(cleanStr(aiAnswer) if cleanStr(aiAnswer) else "none")
            openRefs.append(cleanStr(gtAnswer) if cleanStr(gtAnswer) else "none")
            
        del pixelValues
        torch.cuda.empty_cache()
        
    res = {"avgInferenceTime": np.mean(inferenceTimes), "overallAccuracy": (sum(p==r for p, r in zip(closePreds, closeRefs)) + sum(p==r for p, r in zip(openPreds, openRefs))) / len(datasetInput)}
    if closeRefs: res["Closed"] = {"ACC": accuracy_score(closeRefs, closePreds), "F1": f1_score(closeRefs, closePreds, average='weighted', zero_division=0), "count": len(closeRefs)}
    if openRefs:
        bertRes = evaluate.load("bertscore").compute(predictions=openPreds, references=openRefs, lang="en")
        res["Open"] = {"BLEU-1": evaluate.load("bleu").compute(predictions=openPreds, references=[[r] for r in openRefs], max_order=1)["bleu"], "ROUGE-L": evaluate.load("rouge").compute(predictions=openPreds, references=[[r] for r in openRefs])["rougeL"], "BERTScore": np.mean(bertRes["f1"]), "count": len(openRefs)}
    return res

# ==========================================
# 7. VÒNG LẶP HỌC LIÊN KẾT CHÍNH
# ==========================================
maxRounds = 1; localEpochs = 2
performanceLog = []; cumulativeCommGb = 0.0
totalSamples = sum(dataSizes)

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN FED-FFT INTERNVL3.5-2B (VQA-RAD, {maxRounds} ROUNDS, {localEpochs} EPOCHS/CLIENT, ALPHA=1.0) ===")

for roundIdx in range(1, maxRounds + 1):
    print(f"\n" + "="*50 + f"\n--- COMMUNICATION ROUND {roundIdx} ---\n" + "="*50)
    latestTrainTimes, roundLoss, peakVramRecorded = [], 0, "0.00 GB"

    print("\n [Server] Đang thiết lập cấu trúc In-Place Accumulation trên RAM máy tính...")
    # Lưu backupWeights trên RAM CPU
    backupWeights = {k: v.clone().to(torch.float16) for k, v in globalWeights.items()}

    for k in globalWeights.keys():
        if globalWeights[k].dtype.is_floating_point:
            globalWeights[k] = torch.zeros_like(globalWeights[k], dtype=torch.float32, device=DEVICE_STORAGE)
            
    for i, clientId in enumerate(clientNames):
        # HuggingFace load_state_dict sẽ tự động đẩy trọng số từ CPU vào các phân vùng GPU của Model
        baseModel.load_state_dict(backupWeights, strict=False)
        force_garbage_collect()
        
        t, l, vram = trainLocalClient(baseModel, trainLoaders[i], clientId, roundIdx=roundIdx, epochs=localEpochs, learningRate=1e-5)
        latestTrainTimes.append(t); roundLoss += l; peakVramRecorded = vram
        aggregate_client_weights(baseModel, globalWeights, dataSizes[i] / totalSamples)

    print("\n [Server] Hoàn tất In-Place Aggregation trên CPU. Đưa về Float16...")
    for k in globalWeights.keys():
        if globalWeights[k].dtype.is_floating_point: globalWeights[k] = globalWeights[k].to(torch.float16)
            
    del backupWeights
    force_garbage_collect()
    
    # Push weights mới nhất từ CPU vào GPUs để chuẩn bị Test đánh giá
    baseModel.load_state_dict(globalWeights, strict=False)
    force_garbage_collect()

    evalRes = evaluateGlobalModel(baseModel, testDataset)
    roundAccClosed = evalRes.get("Closed", {}).get("ACC", 0) * 100
    
    roundLog = {
        "Round": roundIdx,
        "Avg Loss": round(roundLoss / NUM_CLIENTS, 4),
        "Overall Acc (%)": round(evalRes["overallAccuracy"] * 100, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(evalRes.get("Closed", {}).get("F1", 0) * 100, 2),
        "Open BLEU-1": round(evalRes.get("Open", {}).get("BLEU-1", 0), 4),
        "Open ROUGE-L": round(evalRes.get("Open", {}).get("ROUGE-L", 0), 4),
        "Open BERTScore": round(evalRes.get("Open", {}).get("BERTScore", 0), 4),
        "MB per Round": round((totalModelParams * 2 * 2 * NUM_CLIENTS) / (1024 * 1024), 2),
        "Avg Train Time (s)": round(np.mean(latestTrainTimes), 2),
        "Peak VRAM": peakVramRecorded
    }
    performanceLog.append(roundLog)

    print(f"\n[VÒNG {roundIdx}] Loss: {roundLog['Avg Loss']:.4f} | Acc: {roundLog['Overall Acc (%)']:.2f}% | Closed Acc: {roundAccClosed:.2f}% | VRAM: {peakVramRecorded}")
    pd.DataFrame(performanceLog).to_csv(logPath, index=False)

    if roundIdx == maxRounds:
        torch.save(globalWeights, checkpointPath)
        print(f"\n   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round {roundIdx}.")

# ==========================================
# 8. BÁO CÁO TỔNG KẾT CHI TIẾT
# ==========================================
print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: FED-FFT INTERNVL3.5 (VQA-RAD, {maxRounds} ROUNDS, 2 EPOCHS, {NUM_CLIENTS} CLIENTS, ALPHA=1.0) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Peak Reserved Memory   : {peakVramRecorded}")
print(f"  + Average Inference Time : {evalRes['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)")
if "Closed" in evalRes:
    print(f"  - CLOSED QUESTIONS ({evalRes['Closed'].get('count', 'N/A')} samples):")
    print(f"    * Accuracy : {evalRes['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {evalRes['Closed']['F1']*100:.2f}%")
if "Open" in evalRes:
    print(f"  - OPEN QUESTIONS ({evalRes['Open'].get('count', 'N/A')} samples):")
    print(f"    * BLEU-1   : {evalRes['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {evalRes['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {evalRes['Open']['BERTScore']:.4f}")

print("\n[3] CONVERGENCE LOG CHI TIẾT")
print(pd.DataFrame(performanceLog).to_string(index=False))

print("\n" + "="*80)
print(" QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


🚀 Phân bổ hệ thống chống OOM: 
 - Huấn luyện: Phân tán tự động qua các GPU (device_map='auto') 
 - Lưu trữ & Tổng hợp (Aggregator): Trên RAM cpu
--- KHỞI TẠO FED-FFT (INTERNVL3.5-2B, 448x448, FP16 - 5 CLIENTS) ---


config.json:   0%|          | 0.00/5.75k [00:00<?, ?B/s]

configuration_internvl_chat.py:   0%|          | 0.00/4.70k [00:00<?, ?B/s]

configuration_intern_vit.py:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

modeling_internvl_chat.py:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

... (47 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


[VÒNG 1] Loss: 0.6501 | Acc: 52.99% | Closed Acc: 72.11% | VRAM: GPU0: 2.4GB | GPU1: 2.1GB

   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round 1.

 FINAL SUMMARY REPORT: FED-FFT INTERNVL3.5 (VQA-RAD, 1 ROUNDS, 2 EPOCHS, 5 CLIENTS, ALPHA=1.0) 

[1] PERFORMANCE & HARDWARE METRICS
  + Peak Reserved Memory   : GPU0: 2.4GB | GPU1: 2.1GB
  + Average Inference Time : 0.2844 giây/ảnh

[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)
  - CLOSED QUESTIONS (251 samples):
    * Accuracy : 72.11%
    * F1-Score : 71.24%
  - OPEN QUESTIONS (200 samples):
    * BLEU-1   : 0.2436
    * ROUGE-L  : 0.3687
    * BERTScore: 0.9048

[3] CONVERGENCE LOG CHI TIẾT
 Round  Avg Loss  Overall Acc (%)  Closed Acc (%)  Closed F1 (%)  Open BLEU-1  Open ROUGE-L  Open BERTScore  MB per Round  Avg Train Time (s)                 Peak VRAM
     1    0.6501            52.99           72.11          71.24       0.2436        0.3687          0.9048      44791.17              667.77 GPU0: 2.4GB | GPU1: 2.1GB

